In [14]:
from dotenv import load_dotenv
from ollama import Client
import os
import pymupdf
import json
from pydantic import BaseModel
load_dotenv()

OLLAMA_API_KEY = os.environ.get("OLLAMA_CLOUD_API_KEY")
PDF_PATH = "../data/resume (9).pdf"
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

class Experience(BaseModel):
    company: str
    title: str
    start_date: str
    end_date: str
    description: str

class Education(BaseModel):
    institution: str
    degree: str
    field_of_study: str
    start_date: str
    end_date: str

class ResumeData(BaseModel):
    first_name: str
    last_name: str
    email: str
    phone: str
    experience_years: int
    skills: list[str]
    job_title: str
    experience: list[Experience]
    education: list[Education]
    summary: str

def _get_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def extract_structured_data(text):
    schema = ResumeData.model_json_schema()
    response = client.chat(
        model="gpt-oss:120b",
        messages=[{"role": "user", "content": f"""
                   You are a data extraction engine.Extract resume data and return only valid JSON from this text: {text}.
                   Return ONLY valid JSON that strictly conforms to this schema: {schema}.
                   Rules:
                   - No markdown
                   - No comments
                   - No extra fields
                   - Use null for missing values
                   - Do not include any text not there in the resume
                   - Do not include any fields that are not in the schema
                   - Use month numbers for dates (e.g. "01" for January)
                   - For skills, only include the skills that are explicitly mentioned in the resume. Do not infer any skills that are not explicitly mentioned.
                   - Use all the skills mentioned in the resume.
                   - For calulating total years of experiece, use all the experience listed in the resume, even if there are overlapping dates. For example, if there are two jobs listed, one from Jan 2020 to Dec 2020 and another from Jun 2020 to Dec 2021, the total years of experience would be 2 years (from Jan 2020 to Dec 2021), not 1 year.
                   """}],
        format="json",
        options={"temperature": 0}
    )
    return response
    

text_from_pdf = _get_text_from_pdf(PDF_PATH)
structured_data =  extract_structured_data(text_from_pdf)
structured_json = structured_data['message']['content']
print(json.dumps(json.loads(structured_json), indent=2))



{
  "first_name": "Zhe",
  "last_name": "Zhu",
  "email": "zhezhu.career@gmail.com",
  "phone": "404-398-8302",
  "experience_years": 7,
  "skills": [
    "HTML",
    "CSS",
    "TypeScript",
    "Angular 2+",
    "Angular Material",
    "C++",
    "Python",
    "MATLAB",
    "Java",
    "Go",
    "Linux",
    "SQL",
    "Hive",
    "Pig",
    "Spark",
    "English",
    "Mandarin",
    "Cantonese"
  ],
  "job_title": "Software Development Engineer",
  "experience": [
    {
      "company": "Amazon Web Services",
      "title": "Software Development Engineer \u2013 AWS EC2 Networking team",
      "start_date": "07/2019",
      "end_date": "Present",
      "description": "Developing distributed route server, written in Python, that calculates routing table and announces it to peer routers via border gateway protocol (BGP) within EC2 network. Developing data-plane software to a fleet that processes a sizeable chunk of all internet traffic."
    },
    {
      "company": "Amazon Web Servi